# CloudDecept Research Analysis Notebook

This notebook analyzes the CloudDecept honeypot dataset for the research paper.

**Data Source**: Oracle Cloud Free Tier deployment (6 weeks)
**Paper**: CloudDecept: Adaptive AI-Powered Cloud Deception
**Authors**: [Anonymous]

---

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data/anonymized')
print(f"Data directory: {DATA_DIR}")

## 1. Load Data

In [ ]:
def load_jsonl(filepath):
    records = []
    with open(filepath, 'r') as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

sessions = load_jsonl(DATA_DIR / 'sessions.jsonl')
commands = load_jsonl(DATA_DIR / 'commands.jsonl')
api_calls = load_jsonl(DATA_DIR / 'cloud_api_calls.jsonl')
mitre = load_jsonl(DATA_DIR / 'mitre_techniques.jsonl')
iocs = load_jsonl(DATA_DIR / 'iocs.jsonl')
summaries = load_jsonl(DATA_DIR / 'session_summaries.jsonl')

print(f"Sessions: {len(sessions)}")
print(f"Commands: {len(commands)}")
print(f"API Calls: {len(api_calls)}")
print(f"MITRE Techniques: {len(mitre)}")
print(f"IOCs: {len(iocs)}")
print(f"Summaries: {len(summaries)}")

## 2. Dataset Overview (Table 1 in Paper)

In [ ]:
overview = {
    'Metric': [
        'Total Sessions',
        'Unique Attacker IPs',
        'Countries',
        'Total Commands',
        'Total Cloud API Calls',
        'Unique MITRE Techniques',
        'Total IOCs Extracted',
        'Avg Session Duration (min)',
        'Median Session Duration (min)',
        'Max Session Duration (min)',
        'Sessions > 10 min',
        'Sessions with MITRE Matches',
        'Sessions with IOCs'
    ],
    'Value': [
        len(sessions),
        sessions['attacker_ip'].nunique(),
        sessions['attacker_country'].nunique(),
        len(commands),
        len(api_calls),
        mitre['technique_id'].nunique(),
        len(iocs),
        f"{sessions['duration_seconds'].mean() / 60:.1f}",
        f"{sessions['duration_seconds'].median() / 60:.1f}",
        f"{sessions['duration_seconds'].max() / 60:.1f}",
        len(sessions[sessions['duration_seconds'] > 600]),
        sessions[sessions['session_id'].isin(mitre['session_id'])].shape[0],
        sessions[sessions['session_id'].isin(iocs['session_id'])].shape[0]
    ]
}
pd.DataFrame(overview).to_string(index=False)

## 3. Intent Distribution (Figure 2)

In [ ]:
intent_counts = sessions['primary_intent'].value_counts()
intent_pct = (intent_counts / len(sessions) * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

intent_counts.plot(kind='bar', ax=ax1, color=sns.color_palette('husl', len(intent_counts)))
ax1.set_title('Primary Intent Distribution (Count)')
ax1.set_xlabel('Intent Category')
ax1.set_ylabel('Number of Sessions')
ax1.tick_params(axis='x', rotation=45)

intent_pct.plot(kind='bar', ax=ax2, color=sns.color_palette('husl', len(intent_pct)))
ax2.set_title('Primary Intent Distribution (%)')
ax2.set_xlabel('Intent Category')
ax2.set_ylabel('Percentage')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/intent_distribution.pdf', bbox_inches='tight')
plt.show()

pd.DataFrame({'Count': intent_counts, 'Percentage': intent_pct})

## 4. Session Duration Analysis (RQ1)

In [ ]:
# Convert to minutes
sessions['duration_min'] = sessions['duration_seconds'] / 60

# Compare adaptive vs static (if we have that column)
if 'adaptations_count' in sessions.columns:
    adaptive = sessions[sessions['adaptations_count'] > 0]
    static = sessions[sessions['adaptations_count'] == 0]
    
    print(f"Adaptive sessions: {len(adaptive)}, avg duration: {adaptive['duration_min'].mean():.1f} min")
    print(f"Static sessions: {len(static)}, avg duration: {static['duration_min'].mean():.1f} min")
    
    from scipy import stats
    t_stat, p_val = stats.ttest_ind(adaptive['duration_min'], static['duration_min'])
    print(f"t-test: t={t_stat:.3f}, p={p_val:.4f}")
    print(f"Significant: {p_val < 0.05}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Duration histogram
axes[0,0].hist(sessions['duration_min'], bins=30, edgecolor='black', alpha=0.7)
axes[0,0].set_xlabel('Session Duration (minutes)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Session Duration Distribution')
axes[0,0].axvline(sessions['duration_min'].mean(), color='red', linestyle='--', label=f'Mean: {sessions["duration_min"].mean():.1f}')
axes[0,0].axvline(sessions['duration_min'].median(), color='green', linestyle='--', label=f'Median: {sessions["duration_min"].median():.1f}')
axes[0,0].legend()

# Duration by intent
sns.boxplot(data=sessions, x='primary_intent', y='duration_min', ax=axes[0,1])
axes[0,1].set_title('Duration by Intent Category')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].set_xlabel('Intent')
axes[0,1].set_ylabel('Duration (minutes)')

# Duration by country (top 10)
top_countries = sessions['attacker_country'].value_counts().head(10).index
country_dur = sessions[sessions['attacker_country'].isin(top_countries)].groupby('attacker_country')['duration_min'].mean().sort_values()
country_dur.plot(kind='barh', ax=axes[1,0])
axes[1,0].set_title('Avg Duration by Country (Top 10)')
axes[1,0].set_xlabel('Avg Duration (minutes)')

# Duration over time
sessions['start_time'] = pd.to_datetime(sessions['start_time'])
sessions.set_index('start_time').resample('D')['duration_min'].mean().plot(ax=axes[1,1])
axes[1,1].set_title('Daily Average Session Duration')
axes[1,1].set_xlabel('Date')
axes[1,1].set_ylabel('Avg Duration (minutes)')

plt.tight_layout()
plt.savefig('../figures/duration_analysis.pdf', bbox_inches='tight')
plt.show()

## 5. Intent Classification Performance (RQ2)

In [ ]:
# Intent confidence analysis
print("Intent Confidence Stats:")
print(sessions['intent_confidence'].describe())

# Confidence by intent
for intent in sessions['primary_intent'].unique():
    subset = sessions[sessions['primary_intent'] == intent]
    print(f"{intent}: n={len(subset)}, mean_conf={subset['intent_confidence'].mean():.3f}")

# Time to first intent prediction (using commands)
first_intent = commands.groupby('session_id').first().reset_index()
first_intent['time_to_intent'] = pd.to_datetime(first_intent['timestamp']).dt.timestamp()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=sessions, x='intent_confidence', hue='primary_intent', multiple='stack', ax=axes[0])
axes[0].set_title('Confidence Distribution by Intent')
axes[0].set_xlabel('Confidence Score')

sns.boxplot(data=sessions, x='primary_intent', y='intent_confidence', ax=axes[1])
axes[1].set_title('Confidence by Intent')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Confidence')

plt.tight_layout()
plt.savefig('../figures/intent_confidence.pdf', bbox_inches='tight')
plt.show()

## 6. MITRE ATT&CK Analysis (RQ3)

In [ ]:
# Technique frequency
tech_counts = mitre['technique_id'].value_counts()
tech_names = mitre.groupby('technique_id')['technique_name'].first()
tech_tactics = mitre.groupby('technique_id')['tactic'].first()
tech_severity = mitre.groupby('technique_id')['severity'].first()

tech_df = pd.DataFrame({
    'Count': tech_counts,
    'Name': tech_names,
    'Tactic': tech_tactics,
    'Severity': tech_severity
}).sort_values('Count', ascending=False)

print("Top MITRE Techniques:")
print(tech_df.head(15).to_string())

# Tactic distribution
tactic_counts = mitre['tactic'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top techniques
top_15 = tech_df.head(15).iloc[::-1]
bars = axes[0].barh(range(len(top_15)), top_15['Count'])
axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels([f"{idx} ({row['Name'][:30]}...)" for idx, row in top_15.iterrows()])
axes[0].set_xlabel('Count')
axes[0].set_title('Top 15 MITRE ATT&CK Techniques')

# Tactic distribution
tactic_counts.plot(kind='bar', ax=axes[1])
axes[1].set_title('MITRE Tactics Distribution')
axes[1].set_xlabel('Tactic')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/mitre_analysis.pdf', bbox_inches='tight')
plt.show()

## 7. IOC Analysis

In [ ]:
ioc_types = iocs['ioc_type'].value_counts()
print("IOC Types:")
print(ioc_types)

# Sample IOCs per type
for ioc_type in ioc_types.index[:5]:
    samples = iocs[iocs['ioc_type'] == ioc_type]['ioc_value'].head(3).tolist()
    print(f"\n{ioc_type} examples:")
    for s in samples:
        print(f"  {s[:80]}...")

fig, ax = plt.subplots(figsize=(10, 5))
ioc_types.plot(kind='bar', ax=ax)
ax.set_title('IOC Types Extracted')
ax.set_xlabel('IOC Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../figures/ioc_types.pdf', bbox_inches='tight')
plt.show()

## 8. Skill Level vs Intent (RQ3)

In [ ]:
# Skill level analysis
skill_intent = pd.crosstab(sessions['skill_level'], sessions['primary_intent'], normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(skill_intent, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Skill Level Distribution by Intent (%)')
axes[0].set_xlabel('Intent')
axes[0].set_ylabel('Skill Level (1-10)')

# Box plot
sns.boxplot(data=sessions, x='primary_intent', y='skill_level', ax=axes[1])
axes[1].set_title('Skill Level by Intent')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Skill Level')
axes[1].set_xlabel('Intent')

plt.tight_layout()
plt.savefig('../figures/skill_intent.pdf', bbox_inches='tight')
plt.show()

print("Avg skill by intent:")
print(sessions.groupby('primary_intent')['skill_level'].mean().sort_values(ascending=False))

## 9. Cloud Provider Targeting (RQ3)

In [ ]:
provider_stats = sessions.groupby('cloud_provider').agg(
    sessions=('session_id', 'count'),
    avg_duration=('duration_min', 'mean'),
    avg_skill=('skill_level', 'mean')
).round(2)
print(provider_stats)

# API calls by provider
api_provider = api_calls.groupby('cloud_provider').size().sort_values(ascending=False)
print("\nAPI Calls by Provider:")
print(api_provider)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

provider_stats['sessions'].plot(kind='bar', ax=axes[0])
axes[0].set_title('Sessions by Cloud Provider')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

provider_stats['avg_duration'].plot(kind='bar', ax=axes[1])
axes[1].set_title('Avg Duration by Provider')
axes[1].set_ylabel('Minutes')
axes[1].tick_params(axis='x', rotation=0)

api_provider.plot(kind='bar', ax=axes[2])
axes[2].set_title('API Calls by Provider')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../figures/cloud_provider.pdf', bbox_inches='tight')
plt.show()

## 10. Temporal Analysis

In [ ]:
sessions['start_time'] = pd.to_datetime(sessions['start_time'])
sessions['hour'] = sessions['start_time'].dt.hour
sessions['day_of_week'] = sessions['start_time'].dt.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Hourly
hourly = sessions.groupby('hour').size()
hourly.plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Sessions by Hour (UTC)')
axes[0,0].set_xlabel('Hour')
axes[0,0].set_ylabel('Count')

# Day of week
dow = sessions.groupby('day_of_week').size()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow.index = [dow_labels[i] for i in dow.index]
dow.plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Sessions by Day of Week')
axes[0,1].set_ylabel('Count')

# Daily trend
daily = sessions.set_index('start_time').resample('D').size()
daily.plot(ax=axes[1,0])
axes[1,0].set_title('Daily Session Count')
axes[1,0].set_xlabel('Date')
axes[1,0].set_ylabel('Sessions')

# Intent over time
intent_time = sessions.set_index('start_time').groupby('primary_intent').resample('D').size().unstack(0)
intent_time.plot(ax=axes[1,1])
axes[1,1].set_title('Intent Categories Over Time')
axes[1,1].set_xlabel('Date')
axes[1,1].set_ylabel('Sessions')
axes[1,1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('../figures/temporal_analysis.pdf', bbox_inches='tight')
plt.show()

## 11. Adaptive vs Static Comparison (RQ1)

In [ ]:
# Compare sessions with vs without adaptations
if 'adaptations_count' in sessions.columns:
    adaptive_sessions = sessions[sessions['adaptations_count'] > 0]
    static_sessions = sessions[sessions['adaptations_count'] == 0]
    
    metrics = {
        'Metric': [
            'Avg Duration (min)',
            'Median Duration (min)',
            'Avg Commands',
            'Avg Skill Level',
            'Sessions > 10 min (%)',
            'Sessions with MITRE (%)'
        ],
        'Adaptive': [
            f"{adaptive_sessions['duration_min'].mean():.1f}",
            f"{adaptive_sessions['duration_min'].median():.1f}",
            f"{adaptive_sessions['commands_count'].mean():.1f}",
            f"{adaptive_sessions['skill_level'].mean():.1f}",
            f"{len(adaptive_sessions[adaptive_sessions['duration_min'] > 10]) / len(adaptive_sessions) * 100:.1f}",
            f"{len(adaptive_sessions[adaptive_sessions['session_id'].isin(mitre['session_id'])]) / len(adaptive_sessions) * 100:.1f}"
        ],
        'Static': [
            f"{static_sessions['duration_min'].mean():.1f}",
            f"{static_sessions['duration_min'].median():.1f}",
            f"{static_sessions['commands_count'].mean():.1f}",
            f"{static_sessions['skill_level'].mean():.1f}",
            f"{len(static_sessions[static_sessions['duration_min'] > 10]) / len(static_sessions) * 100:.1f}",
            f"{len(static_sessions[static_sessions['session_id'].isin(mitre['session_id'])]) / len(static_sessions) * 100:.1f}"
        ]
    }
    pd.DataFrame(metrics).to_string(index=False)
    
    # Statistical tests
    from scipy import stats
    
    print("\nStatistical Tests (Adaptive vs Static):")
    for metric_name, attr in [('Duration', 'duration_min'), ('Commands', 'commands_count'), ('Skill', 'skill_level')]:
        t, p = stats.ttest_ind(adaptive_sessions[attr], static_sessions[attr])
        print(f"  {metric_name}: t={t:.3f}, p={p:.4f}, significant={p < 0.05}")

else:
    print("adaptations_count column not found in sessions")

## 12. Novel Technique Combinations (RQ4)

In [ ]:
# Find session-level technique combinations
session_techniques = mitre.groupby('session_id')['technique_id'].apply(list).reset_index()
session_techniques['combo'] = session_techniques['technique_id'].apply(lambda x: tuple(sorted(x)))

combo_counts = session_techniques['combo'].value_counts()
print(f"Unique technique combinations: {len(combo_counts)}")
print(f"Combinations appearing >1: {(combo_counts > 1).sum()}")

print("\nTop technique combinations:")
for combo, count in combo_counts.head(15).items():
    if len(combo) > 1:
        names = [mitre[mitre['technique_id'] == t]['technique_name'].values[0] for t in combo]
        print(f"  {count}x: {' + '.join([f'{t} ({n[:30]})' for t, n in zip(combo, names)])}")

# Novel combinations (not in standard ATT&CK chains)
known_chains = [
    ('T1526', 'T1530'),  # Discovery -> Collection
    ('T1098', 'T1550.007'),  # Account Manip -> Token Abuse
    ('T1021.004', 'T1564.006'),  # SSH -> Hide Artifacts
]

novel_combos = []
for combo, count in combo_counts.items():
    if len(combo) >= 2:
        combo_set = set(combo)
        is_known = False
        for chain in known_chains:
            if set(chain).issubset(combo_set):
                is_known = True
                break
        if not is_known and count >= 2:
            novel_combos.append((combo, count))

print(f"\nNovel combinations (>=2 occurrences): {len(novel_combos)}")
for combo, count in novel_combos[:10]:
    names = [mitre[mitre['technique_id'] == t]['technique_name'].values[0] for t in combo]
    print(f"  {count}x: {' -> '.join([f'{t}: {n[:40]}' for t, n in zip(combo, names)])}")

## 13. Generate Paper Tables

In [ ]:
# Table 1: Dataset Summary
table1 = pd.DataFrame({
    'Metric': [
        'Deployment Duration',
        'Total Sessions',
        'Unique Attacker IPs',
        'Countries',
        'Total Commands',
        'Cloud API Calls',
        'MITRE Techniques (unique)',
        'IOCs Extracted',
        'Avg Session Duration',
        'Median Session Duration',
        'Adaptive Sessions',
        'Sessions > 10 min'
    ],
    'Value': [
        '6 weeks',
        len(sessions),
        sessions['attacker_ip'].nunique(),
        sessions['attacker_country'].nunique(),
        len(commands),
        len(api_calls),
        mitre['technique_id'].nunique(),
        len(iocs),
        f"{sessions['duration_min'].mean():.1f} min",
        f"{sessions['duration_min'].median():.1f} min",
        len(sessions[sessions.get('adaptations_count', 0) > 0]),
        len(sessions[sessions['duration_min'] > 10])
    ]
})
print(table1.to_latex(index=False, caption='CloudDecept Dataset Summary'))

# Table 2: Intent Classification Performance
table2 = pd.DataFrame({
    'Intent': sessions['primary_intent'].unique(),
    'Sessions': [len(sessions[sessions['primary_intent'] == i]) for i in sessions['primary_intent'].unique()],
    'Avg Confidence': [sessions[sessions['primary_intent'] == i]['intent_confidence'].mean() for i in sessions['primary_intent'].unique()],
    'Avg Duration (min)': [sessions[sessions['primary_intent'] == i]['duration_min'].mean() for i in sessions['primary_intent'].unique()],
    'Avg Skill': [sessions[sessions['primary_intent'] == i]['skill_level'].mean() for i in sessions['primary_intent'].unique()]
}).round(3)
print("\n" + table2.to_latex(index=False, caption='Intent Classification Results'))

# Table 3: Adaptive vs Static
if 'adaptations_count' in sessions.columns:
    table3 = pd.DataFrame({
        'Metric': ['Avg Duration (min)', 'Median Duration (min)', 'Avg Commands', 'Sessions > 10min (%)', 'MITRE Matches (%)'],
        'Adaptive': [
            f"{adaptive_sessions['duration_min'].mean():.1f}",
            f"{adaptive_sessions['duration_min'].median():.1f}",
            f"{adaptive_sessions['commands_count'].mean():.1f}",
            f"{len(adaptive_sessions[adaptive_sessions['duration_min'] > 10]) / len(adaptive_sessions) * 100:.1f}",
            f"{len(adaptive_sessions[adaptive_sessions['session_id'].isin(mitre['session_id'])]) / len(adaptive_sessions) * 100:.1f}"
        ],
        'Static': [
            f"{static_sessions['duration_min'].mean():.1f}",
            f"{static_sessions['duration_min'].median():.1f}",
            f"{static_sessions['commands_count'].mean():.1f}",
            f"{len(static_sessions[static_sessions['duration_min'] > 10]) / len(static_sessions) * 100:.1f}",
            f"{len(static_sessions[static_sessions['session_id'].isin(mitre['session_id'])]) / len(static_sessions) * 100:.1f}"
        ],
        'p-value': [
            f"{stats.ttest_ind(adaptive_sessions['duration_min'], static_sessions['duration_min']).pvalue:.4f}",
            'N/A',
            f"{stats.ttest_ind(adaptive_sessions['commands_count'], static_sessions['commands_count']).pvalue:.4f}",
            'N/A',
            'N/A'
        ]
    })
    print("\n" + table3.to_latex(index=False, caption='Adaptive vs Static Honeypot Comparison'))